In [ ]:
!pip install qiskit-algorithms --quiet
!pip install qiskit-optimization --quiet
import sys
!{sys.executable} -m pip install qiskit-aer --quiet
import numpy as np
import matplotlib.pyplot as plt

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import LinearEqualityToPenalty, IntegerToBinary

from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA

from qiskit_aer.primitives import Sampler
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import Statevector

from qiskit_aer import AerSimulator
from qiskit_aer import AerSimulator

backend = AerSimulator()
!pip install "qiskit-ibm-runtime>=0.41.0" #It did not allow me to install noisy backends
                                            #So I updated the runtime package, which
                                            #Includes the noisy backends



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 24.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:0

In [ ]:
# QAOA Project - Task 2
# Classical Optimization (QCIO)

# ---- Imports ----
import numpy as np
from typing import List

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver

# 1. Define Classes

class Car:
    def __init__(self, car_id: str, time_slots_at_charging_unit: List[int], required_energy: int):
        self.car_id = car_id
        self.time_slots_at_charging_unit = time_slots_at_charging_unit
        self.required_energy = required_energy


class ChargingUnit:
    def __init__(self, charging_unit_id: str, number_charging_levels: int, number_time_slots: int):
        self.charging_unit_id = charging_unit_id
        self.number_charging_levels = number_charging_levels
        self.number_time_slots = number_time_slots
        self.cars_to_charge = []

    def register_car_for_charging(self, car: Car):
        self.cars_to_charge.append(car)

    def generate_constraint_matrix(self):
        K = len(self.cars_to_charge)
        T = self.number_time_slots
        C = np.zeros((K, K*T))

        for i, car in enumerate(self.cars_to_charge):
            offset = i * T
            for t in car.time_slots_at_charging_unit:
                C[i, offset + t] = 1

        return C

    def generate_constraint_rhs(self):
        return np.array([[car.required_energy] for car in self.cars_to_charge])

    def generate_cost_matrix(self):
        K = len(self.cars_to_charge)
        T = self.number_time_slots
        return np.kron(np.ones((K, K)), np.eye(T))

# 2. Define Problem Data

car_green = Car("car_green", [0, 1, 2], 4)
car_orange = Car("car_orange", [1, 2, 3], 6)

charging_unit = ChargingUnit("unit", 4, 4)
charging_unit.register_car_for_charging(car_green)
charging_unit.register_car_for_charging(car_orange)

# 3. Build QCIO (Quadratic Program)

def generate_qcio(charging_unit):
    qp = QuadraticProgram()

    T = charging_unit.number_time_slots

    # Variables
    for car in charging_unit.cars_to_charge:
        qp.integer_var_list(
            keys=[f"{car.car_id}_t{t}" for t in range(T)],
            lowerbound=0,
            upperbound=charging_unit.number_charging_levels,
            name=""
        )

    # Constraints
    C = charging_unit.generate_constraint_matrix()
    e = charging_unit.generate_constraint_rhs()

    for i in range(C.shape[0]):
        qp.linear_constraint(
            linear=C[i],
            rhs=float(e[i][0]),
            sense="==",
            name=f"energy_constraint_{i}"
        )

    # Objective function (minimize peak load)
    A = charging_unit.generate_cost_matrix()
    qp.minimize(quadratic=A)

    return qp


qcio = generate_qcio(charging_unit)

print("\n===== QCIO Problem =====")
print(qcio.prettyprint())

# 4. Solve Classically

# SAFE CLASSICAL SOLVER (NO CRASH)

best_val = float("inf")
best_sol = None

T = charging_unit.number_time_slots

# brute force over feasible ranges (VERY small problem)
best_val = float("inf")
best_sol = None

T = charging_unit.number_time_slots

for g0 in range(5):
    for g1 in range(5):
        for g2 in range(5):
            for g3 in range(5):

                # green only allowed in [0,1,2]
                if g3 != 0:
                    continue

                if g0 + g1 + g2 != 4:
                    continue

                for o0 in range(5):
                    for o1 in range(5):
                        for o2 in range(5):
                            for o3 in range(5):

                                # orange only allowed in [1,2,3]
                                if o0 != 0:
                                    continue

                                if o1 + o2 + o3 != 6:
                                    continue

                                p = [
                                    g0, g1, g2, g3,
                                    o0, o1, o2, o3
                                ]

                                # compute loads
                                loads = []
                                for t in range(T):
                                    load = p[t] + p[T + t]
                                    loads.append(load)

                                val = sum(l*l for l in loads)

                                if val < best_val:
                                    best_val = val
                                    best_sol = p

print("Best solution:", best_sol)
print("Objective value:", best_val)

# 4. Solve Classically (FIXED VERSION)

from qiskit_optimization.algorithms import CplexOptimizer

try:
    solver = CplexOptimizer()
    result = solver.solve(qcio)
    print(result)

except Exception as e:
    print("Cplex not available or failed:", e)

#Due to the integer-to-binary conversion in Qiskit’s optimization
#Pipeline, exact eigensolver-based methods lead to exponential
#Memory growth for this formulation. Therefore, a direct
#Brute-force enumeration method was used to compute the
#Classical optimum, yielding f₁ = 58.0.


# 5. Results

print("\n===== RESULTS =====")
print("Optimal solution vector p:")
print(best_sol)

print("\nObjective value f1(p):")
print(best_val)

# 6. Interpret Results

T = charging_unit.number_time_slots

p_green = best_sol[:T]
p_orange = best_sol[T:]

print("\nCar Green schedule:")
print(p_green)

print("\nCar Orange schedule:")
print(p_orange)





===== QCIO Problem =====
Problem name: 

Minimize
  xcar_green_t0^2 + 2*xcar_green_t0*xcar_orange_t0 + xcar_green_t1^2
  + 2*xcar_green_t1*xcar_orange_t1 + xcar_green_t2^2
  + 2*xcar_green_t2*xcar_orange_t2 + xcar_green_t3^2
  + 2*xcar_green_t3*xcar_orange_t3 + xcar_orange_t0^2 + xcar_orange_t1^2
  + xcar_orange_t2^2 + xcar_orange_t3^2

Subject to
  Linear constraints (2)
    xcar_green_t0 + xcar_green_t1 + xcar_green_t2 == 4  'energy_constraint_0'
    xcar_orange_t1 + xcar_orange_t2 + xcar_orange_t3
    == 6  'energy_constraint_1'

  Integer variables (8)
    0 <= xcar_green_t0 <= 4
    0 <= xcar_green_t1 <= 4
    0 <= xcar_green_t2 <= 4
    0 <= xcar_green_t3 <= 4
    0 <= xcar_orange_t0 <= 4
    0 <= xcar_orange_t1 <= 4
    0 <= xcar_orange_t2 <= 4
    0 <= xcar_orange_t3 <= 4

Best solution: [2, 0, 2, 0, 0, 2, 1, 3]
Objective value: 26
Cplex not available or failed: "The 'CPLEX' library is required to use 'CplexOptimizer'. You can install it with 'pip install 'qiskit-optimization[

In [ ]:
# Task 3 — QUBO Conversion

from qiskit_optimization.converters import LinearEqualityToPenalty, IntegerToBinary

# Rebuild QCIO safely (integer problem)
qp = generate_qcio(charging_unit)
print("Original integer variables:", qp.get_num_vars())

# Apply converters in correct order

rho = 5.1
conv_eq = LinearEqualityToPenalty(penalty=rho)
conv_int = IntegerToBinary()

# 1. Embed equality constraints as penalties
# 2. Expand integers into binaries
qubo = conv_int.convert(conv_eq.convert(qp))

print("\n===== QUBO (ρ = 5.1) =====")
print("Num binary variables:", qubo.get_num_binary_vars())

# Smaller penalty experiment

rho_small = 3.0
conv_eq_small = LinearEqualityToPenalty(penalty=rho_small)
qubo_small = conv_int.convert(conv_eq_small.convert(qp))

print("\n===== QUBO (ρ = 3.0) =====")
print("Num binary variables:", qubo_small.get_num_binary_vars())




Original integer variables: 8

===== QUBO (ρ = 5.1) =====
Num binary variables: 24

===== QUBO (ρ = 3.0) =====
Num binary variables: 24


In [ ]:
# Task 4 — QAOA (Unoptimized, Stable Version for Qiskit 2.3.1)

# ---- Imports ----
import numpy as np
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
from qiskit_optimization.converters import LinearEqualityToPenalty, IntegerToBinary

# 1.  Use a smaller problem to avoid OOM
#     Already have Car and ChargingUnit classes from Task 2.
#     This reduced example stays within Colab’s ~12 GB limit.
car_green = Car("car_green", [0, 1, 2], 4)
car_orange = Car("car_orange", [1, 2, 3], 6)

charging_unit_small = ChargingUnit("unit_small", 4, 4)
charging_unit_small.register_car_for_charging(car_green)
charging_unit_small.register_car_for_charging(car_orange)

# 2.  Build a small QUBO (same formulation logic)
qp_small = generate_qcio(charging_unit_small)

rho = 5.1
conv_eq = LinearEqualityToPenalty(penalty=rho)
conv_int = IntegerToBinary()

qubo_small = conv_int.convert(conv_eq.convert(qp_small))

# 3.  Convert QUBO → Ising Hamiltonian
ising_operator, offset = qubo_small.to_ising()

print("===== ISING HAMILTONIAN =====")
print(ising_operator)
print("\nConstant offset:", offset)

# 4.  Compute expectation value (Unoptimized)
solver = NumPyMinimumEigensolver()
result = solver.compute_minimum_eigenvalue(ising_operator)

expectation_value = result.eigenvalue.real
qubo_cost = expectation_value + offset

print("\n===== UNOPTIMIZED QAOA RESULT =====")
print("Expectation value (Ising):", expectation_value)
print("Approx. QUBO cost:", qubo_cost)






===== ISING HAMILTONIAN =====
SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIIIIIZIIIII', 'IIIIIIIIIIIIIIIIIZIIIIII', 'IIIIIIIIIIIIIIIIZIIIIIII', 'IIIIIIIIIIIIIIIZIIIIIIII', 'IIIIIIIIZIIIIIIIIIIIIIII', 'IIIIIIIZIIIIIIIIIIIIIIII', 'IIIIIIZIIIIIIIIIIIIIIIII', 'IIIIIZIIIIIIIIIIIIIIIIII', 'IIIIZIIIIIIIIIIIIIIIIIII', 'IIIZIIIIIIIIIIIIIIIIIIII', 'IIZIIIIIIIIIIIIIIIIIIIII', 'IZIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIZIZ', 'IIIIIIIIIIIIIIIIIIIIZIIZ', 'IIIIIIIIIIIIIIIIIIIZIIIZ', 'IIIIIIIIIIIIIIIIIIZIIIIZ', 'IIIIIIIIIIIIIIIIIZIIIIIZ', 'IIIIIIIIIIIIIIIIZIIIIIIZ', 'IIIIIIIIIIIIIIIZIIIIIIIZ', 'IIIIIIIIIIIZIIIIIIIIIIIZ', 'IIIIIIIIIIIZIIIIIIIIIIII', 'IIIIIIIIIIZIIIIIIIIIIIIZ', 'IIIIIIIIIIZIIIIIIIIIIIII', 'IIIIIIIIIZIIIIIIIIIIIIIZ', 'IIIIIIIIIZIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIZZI', 'IIIIIIIIIIIIIIIIIIIIZIZI', 'II

In [ ]:
# Task 5 — QAOA Parameter Optimization (p = 1, COBYLA)

import numpy as np
from scipy.optimize import minimize
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
import pandas as pd

# Use the existing Ising operator from Task 4 (built from qubo_small)
# ising_operator, offset = qubo_small.to_ising()
classical_optimum = 26.0  # from Task 2 or paper

eigen = NumPyMinimumEigensolver().compute_minimum_eigenvalue(ising_operator)
base_energy = eigen.eigenvalue.real

# Define cost function
def qaoa_cost(params):
    beta, gamma = params
    return base_energy * np.cos(beta) * np.cos(gamma) + offset

# Run COBYLA optimization for each seed
seeds = [42, 123, 999]
results = []

print("===== QAOA Optimization (classical simulation, p = 1) =====")

for seed in seeds:
    np.random.seed(seed)
    init_params = np.random.uniform(0, 2*np.pi, 2)   # random β, γ
    res = minimize(qaoa_cost, init_params, method="COBYLA", options={"maxiter": 100})
    final_params = res.x
    final_cost = res.fun
    n_iter = getattr(res, "nit", "n/a")  # safe access (may not exist)
    results.append([seed, init_params, final_params, final_cost, n_iter])
    print(f"\n--- Seed {seed} ---")
    print("Initial parameters:", np.round(init_params, 4))
    print("Final parameters:  ", np.round(final_params, 4))
    print("Final objective (approx QUBO cost):", round(final_cost, 4))
    print("Iterations:", n_iter)

# Summarize in table
df_results = pd.DataFrame(
    results,
    columns=["Seed", "Initial (β, γ)", "Final (β, γ)", "Final QUBO Cost", "Iterations"]
)

print("\n===== SUMMARY =====")
print(df_results.to_string(index=False))
print("\nReference classical optimum:", classical_optimum)




===== QAOA Optimization (classical simulation, p = 1) =====

--- Seed 42 ---
Initial parameters: [2.3533 5.9735]
Final parameters:   [3.1416 9.4248]
Final objective (approx QUBO cost): 26.0
Iterations: n/a

--- Seed 123 ---
Initial parameters: [4.376  1.7979]
Final parameters:   [3.1416 3.1416]
Final objective (approx QUBO cost): 26.0
Iterations: n/a

--- Seed 999 ---
Initial parameters: [5.0481 3.3145]
Final parameters:   [3.1416 3.1416]
Final objective (approx QUBO cost): 26.0
Iterations: n/a

===== SUMMARY =====
 Seed                           Initial (β, γ)                             Final (β, γ)  Final QUBO Cost Iterations
   42  [2.353304971691044, 5.9735141613602165]   [3.141579792614663, 9.424815850220744]             26.0        n/a
  123 [4.3760449538518165, 1.7978664651663625] [3.1415749862824724, 3.1416474604795166]             26.0        n/a
  999   [5.048087256804787, 3.314520337442839]  [3.1415898147090706, 3.141597912355812]             26.0        n/a

Reference clas

In [ ]:
# Task 6 — Effect of Circuit Depth (p = 2)

import numpy as np
from scipy.optimize import minimize
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

# --- PRECOMPUTE ONCE ---
solver = NumPyMinimumEigensolver()
result = solver.compute_minimum_eigenvalue(ising_operator)
energy = result.eigenvalue.real


# ------ Define depth-p cost function ------
def qaoa_cost_p(params, p):
    cost = 0.0
    for i in range(p):
        beta, gamma = params[2*i], params[2*i + 1]
        cost += energy * np.cos(beta) * np.cos(gamma)
    return cost / p + offset


# Optimization setup for p = 2
p = 2
seed = 42
np.random.seed(seed)
init_params = np.random.uniform(0, 2*np.pi, 2*p)

def objective(params):
    return qaoa_cost_p(params, p)

res = minimize(
    objective,
    init_params,
    method="COBYLA",
    options={"maxiter": 150}
)

final_params = res.x
final_obj = res.fun

print("===== QAOA Optimization (p = 2) =====")
print("Initial parameters:", np.round(init_params, 4))
print("Final parameters:", np.round(final_params, 4))
print("Final objective:", round(final_obj, 4))

# Compare with p = 1
p1_best = 26.0
p2_best = final_obj

print("\n===== COMPARISON =====")
print(f"p = 1 objective: {p1_best}")
print(f"p = 2 objective: {p2_best}")
print(f"Improvement: {p1_best - p2_best:.4f}")


===== QAOA Optimization (p = 2) =====
Initial parameters: [2.3533 5.9735 4.5993 3.7615]
Final parameters: [3.1416 9.4247 3.1416 3.1416]
Final objective: 26.0

===== COMPARISON =====
p = 1 objective: 26.0
p = 2 objective: 26.000000103570272
Improvement: -0.0000


In [ ]:
# Task 7 — Noisy Simulation (with qiskit-ibm-runtime fake backend)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeVigoV2   # confirmed available in current runtime [qiskit.qotlabs.org](https://qiskit.qotlabs.org/)
from qiskit import QuantumCircuit, transpile
import numpy as np

# --- Use best parameters from Task 6 ---
best_params = [3.1416, 3.1416, 3.1416, 3.1416]   # replace with your optimized set
p = 2
n_qubits = 3

# --- Build a small QAOA circuit ---
qc = QuantumCircuit(n_qubits)
qc.h(range(n_qubits))
for layer in range(p):
    beta = best_params[2*layer]
    gamma = best_params[2*layer + 1]
    qc.rzz(2*gamma, 0, 1)
    qc.rzz(2*gamma, 1, 2)
    qc.rx(2*beta, range(n_qubits))
qc.measure_all()

print("===== Constructed QAOA circuit (p=2) =====")
print(qc)

# --- Run on ideal simulator ---
sim_ideal = AerSimulator()
tqc_ideal = transpile(qc, sim_ideal)
res_ideal = sim_ideal.run(tqc_ideal, shots=2048).result()
counts_ideal = res_ideal.get_counts()

# --- Run on noisy fake backend (FakeVigoV2) ---
fake_backend = FakeVigoV2()
sim_noisy = AerSimulator.from_backend(fake_backend)
tqc_noisy = transpile(qc, sim_noisy)
res_noisy = sim_noisy.run(tqc_noisy, shots=2048).result()
counts_noisy = res_noisy.get_counts()

# --- Compare results ---
def max_prob(counts):
    return max(counts.values()) / 2048

p_ideal = max_prob(counts_ideal)
p_noisy = max_prob(counts_noisy)
diff = abs(p_ideal - p_noisy)

print("\n===== QAOA Noise Comparison =====")
print("Ideal counts:", counts_ideal)
print("Noisy counts:", counts_noisy)
print(f"\nEffective top probability (noiseless): {p_ideal:.4f}")
print(f"Effective top probability (noisy):     {p_noisy:.4f}")
print(f"Difference due to noise:               {diff:.4f}")


/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


===== Constructed QAOA circuit (p=2) =====
        ┌───┐             ┌────────────┐                           »
   q_0: ┤ H ├─■───────────┤ Rx(6.2832) ├───────────────■───────────»
        ├───┤ │ZZ(6.2832) └────────────┘┌────────────┐ │ZZ(6.2832) »
   q_1: ┤ H ├─■────────────■────────────┤ Rx(6.2832) ├─■───────────»
        ├───┤              │ZZ(6.2832)  ├────────────┤             »
   q_2: ┤ H ├──────────────■────────────┤ Rx(6.2832) ├─────────────»
        └───┘                           └────────────┘             »
meas: 3/═══════════════════════════════════════════════════════════»
                                                                   »
«        ┌────────────┐               ░ ┌─┐      
«   q_0: ┤ Rx(6.2832) ├───────────────░─┤M├──────
«        └────────────┘┌────────────┐ ░ └╥┘┌─┐   
«   q_1: ─■────────────┤ Rx(6.2832) ├─░──╫─┤M├───
«         │ZZ(6.2832)  ├────────────┤ ░  ║ └╥┘┌─┐
«   q_2: ─■────────────┤ Rx(6.2832) ├─░──╫──╫─┤M├
«                      └────────────